In [ ]:
import os
os.environ['GIADA_IE_P7_REVISION']='f696eef26b4e62bfe22040af2878b9fe12b9828a'
import importlib.metadata, json, os, shutil, subprocess, sys
from pathlib import Path
WORK=Path('/kaggle/working/giada_ie_p7');WORK.mkdir(parents=True,exist_ok=True)
REPO,TEACHER=WORK/'giada',WORK/'neuron_as_deep_net'
BRANCH='codex/surrogate-validity-audit'
if not REPO.exists():subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/Zagred47/giada.git',str(REPO)],check=True)
REF=os.environ.get('GIADA_IE_P7_REVISION',BRANCH)
subprocess.run(['git','-C',str(REPO),'fetch','--depth','1','origin',REF],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach','FETCH_HEAD'],check=True)
REVISION=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
if not TEACHER.exists():subprocess.run(['git','clone','--depth','1','https://github.com/SelfishGene/neuron_as_deep_net.git',str(TEACHER)],check=True)
TEACHER_REV='074c4666300a8ad246601dab179a97a6942f0f29'
subprocess.run(['git','-C',str(TEACHER),'fetch','--depth','1','origin',TEACHER_REV],check=True)
subprocess.run(['git','-C',str(TEACHER),'checkout','--detach','FETCH_HEAD'],check=True)
try: installed=importlib.metadata.version('NEURON').split('+')[0]
except importlib.metadata.PackageNotFoundError: installed=None
if installed!='8.2.7':subprocess.run([sys.executable,'-m','pip','install','--quiet','neuron==8.2.7'],check=True)
assert importlib.metadata.version('NEURON').split('+')[0]=='8.2.7'
assert shutil.which('nrnivmodl') or (Path(sys.executable).parent/'nrnivmodl').is_file()
print({'revision':REVISION,'teacher':TEACHER_REV,'training':False,'inputs_required':0})


In [ ]:
import sys,json,numpy as np
sys.path.insert(0,str(REPO))
from src.giada_teacher import ie_native_operational_confirmation as p
from neuron import h,load_mechanisms
out=Path('/kaggle/working/diagnostic');out.mkdir()
spec,cfg,ca,c3,base=p.contract(REPO)
build=p.t33.compile_combined(REPO,TEACHER,out);load_mechanisms(str(build.resolve()))
n=p.t33._native_case(h,cfg,base,ca,c3,spec['calibration'])
s=p.t33.shadow_case(h,cfg,c3,spec['calibration'],n)
print('SHADOW',json.dumps({k:v for k,v in s.items() if not isinstance(v,np.ndarray)}))
for j,params in enumerate(c3['synapses']):
 r='AMPA' if params['mechanism']=='ProbAMPANMDA_EMS' else 'GABAA'
 a=n['native_ab'][:,j,p.t33.AB_NAMES.index('A_'+r)]
 events=cfg['schedules_ms'][spec['calibration']['schedule']][params['id']]
 rise=n['parameters'][r][0]
 jumps=a[1:]-a[:-1]*np.exp(-.1/rise)
 hits=np.where(jumps>1e-7)[0]+1
 factor=n['parameters'][r][2]*cfg['synapse_weight_ns']
 print('PHASE',params['id'],events,[(float(n['clock'][i]),float(jumps[i-1]),float(-rise*np.log(jumps[i-1]/factor))) for i in hits[:8]])
np.savez_compressed(out/'calibration_native.npz',**{k:v for k,v in n.items() if isinstance(v,np.ndarray)})
(out/'metadata.json').write_text(json.dumps({k:v for k,v in n.items() if not isinstance(v,np.ndarray)}))
